In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df.drop(columns="churn value"), # new df, not modify
    df["churn value"],
    test_size = 0.2,
    random_state = 42, # the answer
    stratify=df["churn value"] # preserves 73.5%, 26.5% split
)

# over n iterations, take a unique 1/n sample fold for validation at each iteration, take averge of all validations
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42) 

print(f"Train size: {X_train.shape[0]}")
print(f"Test size:  {X_test.shape[0]}")
print(f"\nClass balance in train:\n{y_train.value_counts(normalize=True).round(3)}")
print(f"\nClass balance in test:\n{y_test.value_counts(normalize=True).round(3)}")

In [ ]:
pipeline = Pipeline([
    ("scalar", StandardScaler()),
    ("model", LogisticRegression(
        class_weight = "balanced",
        max_iter = 1000,
        random_state = 42
    ))
])

metrics = ["f1", "roc_auc", "precision", "recall"]
cross_validation_results = cross_validate(pipeline, X_train, y_train, cv=skf, scoring=metrics)

for metric in metrics:
    scores = cross_validation_results[f"test_{metric}"]
    print(f"{metric.upper()} per fold: {np.round(scores, 3)}")
    print(f"Mean: {scores.mean():.3f}")
    print(f"Standard Deviation: {scores.std():.3f}\n")

In [ ]:
y_prediction_cross_validation = cross_val_predict(pipeline, X_train, y_train, cv=skf)

confusion_matrix = confusion_matrix(y_train, y_prediction_cross_validation)
display = ConfusionMatrixDisplay(confusion_matrix=confusion_matrix, display_labels=["Retained", "Churned"])
display.plot(cmap="coolwarm")
plt.title("Logistic Regression")
plt.tight_layout()
plt.show()

print("\nClassification Report:")
print(classification_report(y_train, y_prediction_cross_validation, target_names=["Retained", "Churned"]))